# AXI test 34: does the repair rule have to be coded? (Gemini 3.8 Flash, second seed, the rule in the prompt, repair on or off)
Pre-registered page: `docs/GEMINI_RULEHINT_TEST.md` (locked before this run). The architecture is Rajnish Choubey's.

Five arms of **400 requests** each on the same hard job as the 4,000-request article (a map of services and connections), on a **second seed** that has never been run: `repair` (the framework as in the article), `hint_gate` (the rule stated in the prompt, only the check, no repair), `hint_repair` (rule stated and repair on), `blind` and `gate` (the two contrasts).

* **No GPU.** Use the free CPU runtime. **Keep this tab open while it runs** (about 30 to 90 minutes of calls).
* **Your key** lives in Colab Secrets (Cell 1). It is never printed or saved in a file. Never paste it in the chat.
* **This test costs real money (about 2 to 4 US dollars) and this notebook never turns billing on.** You do that in Google AI Studio. Read the price of **Gemini 3.8 Flash** on the AI Studio price page (dollars per 1,000,000 input tokens and per 1,000,000 output tokens; thinking tokens are billed as output) and type the two numbers and a **budget in dollars** (I suggest **6**) into Cell 4. The notebook refuses the next call once the saved answers have cost the budget. Everything finished is saved; you can raise the budget and carry on.

### Cell 1: put your key into Colab Secrets (once; no code to run)
Same as before: the key icon in the left bar -> *Add new secret* -> name `GEMINI_API_KEY` -> paste the key -> switch *Notebook access* on.

### Cell 2: get the code from GitHub and install the Gemini library (about 30 seconds)

In [ ]:
!rm -rf /content/axi && git clone -q --depth 1 https://github.com/gaurshalendra2412/axi-runtime /content/axi
!pip -q install -U google-genai
import os
assert os.path.exists("/content/axi/python/axi/experiments/gemini_rulehint.py"), "test 34 is not on GitHub yet"

### Cell 3: connect Google Drive (a pop-up asks permission: tap Connect / Allow). This is where your progress and every reply are kept

In [ ]:
import sys, os
sys.path.insert(0, "/content/axi/python")
from colab.colab_persist import open_store
store = open_store("/content/out", "axi_gemini_rulehint")

### Cell 4: settings, and connect to the model. Fill in the four lines at the top (PAID, the two prices, the budget)

In [ ]:
import json, time
from collections import deque
from google.colab import userdata, files
from axi.experiments.api_llm import QuotaStop, ServiceBusy, BudgetStop, build_gemini_llm
from axi.experiments.million import run_million, print_million, smoke_report
from axi.experiments import million_hard as H
from axi.experiments import gemini_rulehint as R

PAID = None        # True ONLY after you turned billing on for this key in Google AI Studio.
PRICE_IN = None    # dollars per 1,000,000 INPUT tokens of Gemini 3.8 Flash, from the AI Studio price page
PRICE_OUT = None   # dollars per 1,000,000 OUTPUT tokens (thinking tokens are billed as output), from the same page
BUDGET_USD = None  # the most you allow this test to cost, in dollars (I suggest 6)

assert PAID is True, "Set PAID = True after turning billing on in AI Studio (this test needs a paid key)."
assert PRICE_IN and PRICE_OUT, "Type PRICE_IN and PRICE_OUT (dollars per million tokens) from the AI Studio price page."
assert BUDGET_USD, "Type BUDGET_USD (I suggest 6)."
try: KEY = userdata.get("GEMINI_API_KEY")
except Exception: raise SystemExit("No key found. Do Cell 1: add a Secret named GEMINI_API_KEY and switch Notebook access on.")
CACHE = "axi_gemini_rulehint_cache.jsonl"
print("reply file:", store.restore(CACHE))
llm, MODEL = build_gemini_llm(KEY, R.RH_MODEL, store.path(CACHE), daily_limit=None, min_interval=0.3, on_write=lambda: store.persist(CACHE), thinking=None,
                              budget_usd=BUDGET_USD, price_in=PRICE_IN, price_out=PRICE_OUT)
META = dict(model=MODEL, thinking=None)
print("model:", MODEL, "| budget: $%.2f" % BUDGET_USD, "| already spent with these settings: $%.2f" % (llm.spent_usd() or 0))

def go(fn):
    try: return fn()
    except QuotaStop as e:
        print("\nSTOPPED FOR NOW: " + str(e))
        if isinstance(e, ServiceBusy): print("Google's server is busy. Wait at least 30 minutes, then run THIS SAME CELL once.")
        elif isinstance(e, BudgetStop): print("The budget is spent. Tell me before you raise it.")
        return None

def make_progress():
    T0, recent = time.time(), deque(maxlen=100)
    def cb(t, n, rec):
        recent.append(rec)
        if t % 50 == 0 or t == n:
            k = len(recent); sp = llm.spent_usd()
            print(f"step {t}/{n} | {(time.time()-T0)/60:.0f} min | spent {('$%.2f' % sp) if sp is not None else '-'} | last {k}: exact {100*sum(r['exact'] for r in recent)/k:.0f}%, mean distance {sum(r['distance'] for r in recent)/k:.2f}, {sum(r.get('asks', 1) - 1 for r in recent)} re-asks")
    return cb

def run_arm(arm, steps=R.RH_STEPS, scripted=0, tag=""):
    pol, hint = R.ARMS[arm]; name = "axi_rulehint_%s%s" % (arm, tag)
    print("\n=== arm %s (policy %s, rule stated in the prompt: %s) ===" % (arm, pol, hint), "| progress file:", store.restore(name + ".jsonl"))
    def persist_cb(inner):
        def cb(t, n, rec):
            inner(t, n, rec)
            if t % 25 == 0 or t == n: store.persist(name + ".jsonl")
        return cb
    res = go(lambda: run_million(llm, R.RH_SEED, steps, pol, store.path(name + ".jsonl"), meta=META, hint_rules=hint, progress=persist_cb(make_progress()), max_tokens=H.HARD_TOKEN_CAP,
                                 stream="HardStream", reasks=H.HARD_REASKS, degree_threshold=H.HARD_DEGREE, scripted=scripted))
    if res and res["stopped"] == "token_cap": print("STOPPED by the token cap; tell me before you raise it."); return None
    return res

### Cell 5: quick check, 20 real calls (a few cents). The program answers the first 200 requests itself, then 20 calls go to the model. If it says NOTE, read it before going on

In [ ]:
smoke = run_arm("repair", steps=H.HARD_WARM + 20, scripted=H.HARD_WARM, tag="_check")
if smoke:
    live = smoke["rows"][H.HARD_WARM:]; sm = smoke["summary"]
    print(f"\ncheck: {len(live)} calls to the model: exact {sum(r['exact'] for r in live)}, refused {sum(not r['parsed'] for r in live)}, {sm['tokens']:,} tokens sent")
    rep = smoke_report(sm, R.RH_STEPS, PRICE_IN, PRICE_OUT, think_limit=H.HARD_THINKING_LIMIT, lowest=H.HARD_LOWEST_THINKING, budget_usd=BUDGET_USD, minutes_limit=H.HARD_MINUTES_LIMIT, token_cap=H.HARD_TOKEN_CAP)
    if rep["steps"] == 0: print("NOTE:", rep["advice"][0])
    else:
        print(f"{rep['tokens_per_step']:.0f} tokens a step, {rep['seconds_per_step']:.2f} s a step: one arm of {R.RH_STEPS} steps would be about ${rep['projected_cost']:.2f}" if rep["projected_cost"] is not None else "")
        for a in rep["advice"]: print("NOTE:", a)
    for r in live[:5]: print(r["t"], r["kind"], "|", repr(r["text"])[:90])

### Cell 6: RUN. The five arms, one after the other (400 requests each). A line is printed every 50 steps; progress and every reply are saved to Drive. If it stops (budget, server busy, tab closed), run Cells 2 to 4 and this cell again; it carries on

In [ ]:
runs = {}
for arm in R.ARM_ORDER:
    r = run_arm(arm)
    if not r: break
    runs[arm] = r
    print_million(r["summary"], "arm %s" % arm)
if len(runs) == len(R.ARM_ORDER):
    R.print_arms(runs, PRICE_IN, PRICE_OUT)
    R.print_verdict_rulehint(R.verdict_rulehint(runs, PRICE_IN, PRICE_OUT))
    out = store.path("gemini_rulehint.json")
    json.dump(dict(model=MODEL, seed=R.RH_SEED, price_in=PRICE_IN, price_out=PRICE_OUT, arms=runs), open(out, "w")); store.persist("gemini_rulehint.json")
    print("spent in all: $%.2f" % llm.spent_usd())
    files.download(out); files.download(store.path(CACHE))
else: print("Not all five arms are finished yet: run this cell again after the note above.")

### Done
Two files download: `gemini_rulehint.json` (this is the one I need) and the reply file `axi_gemini_rulehint_cache.jsonl`. If a download does not start, both are in your Google Drive, in the folder `axi_gemini_rulehint`. Upload `gemini_rulehint.json` to the chat, with a screenshot of the printed table and the reading-rule lines. Do not paste any keys. When you are done, you can switch billing off in AI Studio; nothing here does it for you.